# NB_hospital_daily_summary

Builds `adb_caresync.gold.hospital_daily_summary` (one row per hospital per test day) from `silver.lab_results` per the STTM `sttm_silver_to_gold.xlsx`, enriched with hospital name and calendar attributes, plus a day-over-day trend via LAG(). Full rebuild on every run.

In [0]:
%sql
-- Full rebuild: one row per hospital per test_date (driving table silver.lab_results, LEFT JOINs only)
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volumes, abnormal/critical rates and day-over-day trend. Source: STTM silver->gold.'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                  AS summary_date,
    COUNT(*)                                   AS total_tests,
    COUNT(DISTINCT patient_id)                 AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                  AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_trend AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(t.hospital_id AS INT)                                                   AS hospital_id,
  h.hospital_name                                                              AS hospital_name,
  t.summary_date                                                               AS summary_date,
  c.day_name                                                                   AS day_name,
  c.is_weekend                                                                 AS is_weekend,
  CAST(t.total_tests AS INT)                                                   AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                        AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                           AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                           AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                           AS critical_test_count,
  CAST(ROUND(t.abnormal_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(t.critical_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                          AS prev_day_total_tests,
  CASE WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
       ELSE CAST(ROUND((t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                                                          AS test_volume_change_pct
FROM with_trend t
LEFT JOIN adb_caresync.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
%sql
-- Grain must be unique on (hospital_id, summary_date); test totals must reconcile with silver.lab_results
SELECT
  COUNT(*)                                                     AS gold_rows,
  COUNT(DISTINCT hospital_id, summary_date)                    AS distinct_grain,
  SUM(total_tests)                                             AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync.silver.lab_results)       AS silver_total_tests,
  COUNT_IF(hospital_name IS NULL)                              AS missing_hospital_name,
  COUNT_IF(day_name IS NULL)                                   AS missing_calendar_match,
  COUNT_IF(prev_day_total_tests IS NULL)                       AS first_day_rows,
  COUNT(DISTINCT hospital_id)                                  AS hospitals
FROM adb_caresync.gold.hospital_daily_summary;

In [0]:
# Return the gold table's record count to the caller (e.g. a job task or dbutils.notebook.run)
updated_df = spark.table("adb_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))